A random field gives every position in space a random value, with values at nearby positions alike. Random fields describe shadow fading in radio channels, the hydraulic conductivity of aquifers, the height of an ocean surface and many other quantities that vary in space without a fixed pattern. This tutorial explains what they are, how they are described by their autocorrelation function, and how the sum-of-sinusoids method of Quadriga-Lib generates them at any position, in one, two or three dimensions.

The code runs unchanged in MATLAB R2024a or newer and in GNU Octave 8.4 or newer; the outputs on this page were made with Octave. It needs Quadriga-Lib 0.13 or newer with its MATLAB/Octave interface, the package folder `+quadriga_lib` on the path; no toolbox or Octave package is needed. See the setup on the [tutorial overview](tutorials.html#setup).

In [ ]:
disp(version())                 % version of MATLAB or Octave
disp(quadriga_lib.version())    % version of Quadriga-Lib

If `quadriga_lib.version` is not found or the version is older than 0.13, the folder that contains `+quadriga_lib` is not on the path, or the path leads to an older build: add the folder of the current build with `addpath`, or update the library as described in the setup. A suffix `_AVX2` means that the fast vector code of the library is active.

## A first random field

Four lines give a map of 200 m × 200 m. The values change smoothly: values 10 m apart have a correlation of only 0.37, which makes 10 m the *decorrelation distance* of the generator; patches of high or low values are about twice that size. The terms used here, a *preset* and a *seed*, are explained further down.

In [ ]:
x = 0:0.5:199.5;                                              % grid points in m, x and y; [1, 400]
[freq, acf, dist] = quadriga_lib.sos_preset('Comb256', 10);   % frequencies [256, 3] in 1/m for a decorrelation distance of 10 m
phase = quadriga_lib.sos_init_phases(size(freq, 1), 1, 42);   % phases [256, 1] in rad: one generator, seed 42
field = quadriga_lib.sos_map(freq, phase, x, x);              % values on the grid [400, 400], rows are y, columns are x

figure('Position', [100, 100, 640, 480]);
imagesc(x, x, field); axis xy; axis image;
cb = colorbar; ylabel(cb, 'value');
xlabel('x in m'); ylabel('y in m');

## What a random field is

A random field is a random function of position [9, Sec. 1]: at every position $\mathbf{x}$ it has a random value $s(\mathbf{x})$, and the values at different positions are related. A random process is the special case of a one-dimensional index, usually time. The map above is one *realization* of the field; another seed gives another map with the same statistics. Statements such as "the values are Normal distributed with zero mean and unit variance" are about the *ensemble*, the set of all realizations, not about one map.

Two simplifications are common and used here. A field is *stationary* (homogeneous) if its statistics are the same everywhere, and *isotropic* if the relation between two positions depends only on their distance, not on the direction [1, Sec. 4.1].

Random fields appear wherever a quantity varies in space without a fixed pattern:

- Radio channels: shadow fading, the slow change of the received power behind buildings and trees, and the other large-scale parameters of channel models such as the 3GPP model [2], [3].
- Geostatistics and hydrology: the hydraulic conductivity of aquifers [4].
- Ocean waves: the height of the sea surface, built from many waves [5].
- Optics: the phase disturbance of light passing through turbulent air [6].
- Materials: two-phase composite materials such as alloys and microemulsions, modeled as level cuts of a Gaussian random field [7].

## The autocorrelation function

How quickly the values change with the position is described by the autocorrelation function (ACF). For a stationary, isotropic field with zero mean and unit variance it is the expected product of the values at two positions a distance $d$ apart,

$$\rho(d) = \mathrm{E}\{\, s(\mathbf{x})\, s(\mathbf{x} + \mathbf{d}) \,\}, \qquad |\mathbf{d}| = d .$$

It starts at $\rho(0) = 1$ and falls towards 0 for positions far apart. The *decorrelation distance* $d_0$ is the distance at which it has fallen to $1/e \approx 0.37$ [2]; this is the correlation distance of the exponential ACF of 3GPP [3, clause 7.4.4]. The exponential decay of the shadow-fading correlation with distance was found in measurements by Gudmundson [8], who described it by the correlation at a reference distance instead, e.g. 0.82 at 100 m. Other sources use other lengths for the same idea, such as the distance at which the correlation falls to 1/2 [14], the length parameter of a kernel [1, Sec. 4.2], or the integral scale in geostatistics [13]; when comparing numbers from other sources, check which one is meant.

Quadriga-Lib has three ACF shapes. `Exp` is the exponential $\rho(d) = e^{-d/d_0}$ of 3GPP [3, clause 7.4.4]. `Gauss` is $\rho(d) = e^{-(d/d_0)^2}$. `Comb` follows `Gauss` up to $d_0$ and `Exp` beyond.

In [ ]:
names = {'Exp256', 'Gauss256', 'Comb256'};
styles = {'-', '-', '--'};
widths = [1.5, 1.5, 2];
figure('Position', [100, 100, 640, 360]); hold on;
for k = 1:3
    [f, a, d] = quadriga_lib.sos_preset(names{k}, 10);
    plot(d, a, styles{k}, 'LineWidth', widths(k));
end
plot([0, 40], exp(-1) * [1, 1], '--', 'Color', [0.5, 0.5, 0.5], 'LineWidth', 0.8);
plot([10, 10], [0, 1], '--', 'Color', [0.5, 0.5, 0.5], 'LineWidth', 0.8);
xlim([0, 40]); xlabel('distance d in m'); ylabel('ACF'); grid on; box on;
legend('Exp', 'Gauss', 'Comb');

The shape near $d = 0$ decides how smooth the field is. The exponential ACF has a kink at the origin: the field is continuous but rough, it changes even over very short distances. The Gaussian ACF is flat at the origin and gives a smooth field [1, Sec. 4.2.1]. `Comb` combines a smooth field at short distances with the exponential decay of 3GPP at large ones.

The ACF has a partner in the frequency domain: the ACF and the power spectral density of a stationary field are a Fourier pair (Wiener-Khinchin theorem) [1, Sec. 4.2.1]. The spectrum tells how much of the variation of the field comes from each spatial frequency, i.e. from waves of each length and direction. This is the key to the method below.

## How random fields are generated

There are two families of methods:

| Family | Idea | Examples | Strength | Cost |
|---|---|---|---|---|
| Matrix methods | Build the covariance matrix of all positions and factor it | Cholesky decomposition [9, Sec. 2.1] | Exact at the given positions | Memory grows with the square of the number of positions, time up to the cube; new positions need a new factorization |
| Sums of waves | Add up waves whose spectrum follows the target | FFT on a grid with circulant embedding [9, Sec. 2.2], randomization method [4, Sec. 2.2.2], sum of sinusoids [2] | Fast, little memory | ACF mostly approximate (circulant embedding is exact on its grid); FFT grids are periodic and bounded |

The sum-of-waves methods differ in three choices: where the frequencies come from (a regular grid, random draws, or a fit), whether the amplitudes are fixed or random, and whether the sum is evaluated by FFT on a grid or directly at each position. Quadriga-Lib uses fixed amplitudes, fitted frequencies and direct evaluation [2]. Each of its properties below follows from these choices.

## The sum-of-sinusoids method

A generator with $N$ sinusoids gives at position $\mathbf{x}$ the value

$$s(\mathbf{x}) = \sqrt{\frac{2}{N}} \sum_{n=1}^{N} \cos\!\left( 2\pi\, \mathbf{f}_n \cdot \mathbf{x} + \varphi_n \right)$$

[2], [11]. The frequency vectors $\mathbf{f}_n$ are samples of the spectrum: they set the ACF, which for random phases is $\frac{1}{N}\sum_n \cos(2\pi\, \mathbf{f}_n \cdot \mathbf{d})$. Quadriga-Lib fits them to the target ACF and ships the results as *presets*. The phases $\varphi_n$, drawn uniformly at random, select one realization. Each column of the phase array is one generator: columns with different phases give independent fields with the same ACF [2, Sec. II], so for one quantity they are its realizations. Since the value is a sum of many independent terms, it is close to Normal distributed (central limit theorem) [11].

The formula is short enough to evaluate by hand. The cell below does so in a few lines of MATLAB for five random positions and compares the result with `sos_val`, the library function that evaluates a generator:

In [ ]:
% five positions {x, y, z} in m; [5, 3]; MATLAB and Octave draw different numbers for the same seed
rng(1); pos = 100 * rand(5, 3);
N = size(freq, 1);
s_formula = sqrt(2 / N) * sum(cos(2 * pi * pos * freq.' + phase(:, 1).'), 2);   % [5, 1]
s_lib = quadriga_lib.sos_val(freq, phase, pos);                                  % [5, 1]
fprintf([repmat('%9.4f', 1, 5), '\n'], [s_formula, s_lib]);              % row 1: formula, row 2: library

With few sinusoids the values are not yet Normal distributed: a sum of $N$ cosines with amplitude $\sqrt{2/N}$ can never exceed $\sqrt{2N}$. The histograms show the values of 20 000 realizations at one position, for the first sinusoid alone, the first 4, and all 256 sinusoids of the preset. The density axis is logarithmic, so that the tails are visible: in the middle, 4 sinusoids already look Normal, but their values stop at $\sqrt{8} \approx 2.8$, where the Normal distribution goes on.

In [ ]:
edges = linspace(-4.5, 4.5, 61);                     % 60 bins
centers = (edges(1:end-1) + edges(2:end)) / 2;
v = linspace(-4.5, 4.5, 200);
n_list = [1, 4, 256];
figure('Position', [100, 100, 1000, 300]);
for k = 1:3
    n = n_list(k);
    ph = quadriga_lib.sos_init_phases(n, 20000, 3);                  % 20 000 realizations; [n, 20000]
    vals = quadriga_lib.sos_val(freq(1:n, :), ph, zeros(1, 3));      % value at the origin; [1, 20000]
    counts = histc(vals, edges);                                     % last element: values equal to 4.5
    density = [counts(1:59), counts(60) + counts(61)] / (numel(vals) * (edges(2) - edges(1)));
    density(density == 0) = NaN;                                     % empty bins: no bar on the log axis
    subplot(1, 3, k); hold on;
    bar(centers, density, 1, 'BaseValue', 1e-4, 'FaceColor', [0.55, 0.7, 0.9], 'EdgeColor', 'none');
    plot(v, exp(-v.^2 / 2) / sqrt(2 * pi), 'k', 'LineWidth', 1);
    set(gca, 'YScale', 'log'); xlim([-4.5, 4.5]); ylim([1e-4, 3]); box on;
    title(sprintf('N = %d', n)); xlabel('value');
    if k == 1, ylabel('density'); end
end

The method has clear strengths:

- Direct evaluation: the value at a position is computed from the formula alone. Unlike FFT methods, which compute the whole field on a grid of points and keep it in memory [2, Table I], any position can be evaluated at any time, in 1, 2 or 3 dimensions, without bounds.
- Little memory: a generator is $N$ frequency vectors and $N$ phases, independent of the size of the area.
- Repeatable: the same frequencies and phases always give the same field.
- Isotropic: the frequency vectors point in all directions.

And costs:

- The ACF is only approximated, and only over the distances the preset was fitted to (up to 5 $d_0$). Beyond that, the ACF of the generator does not fall to zero but keeps a small residual with rms $1/\sqrt{2N}$, 0.044 for 256 sinusoids.
- The computing time per position grows with $N$.
- The values are approximately, not exactly, Normal distributed.

## Rough and smooth fields

`sos_randn` is the shortest way to spatially correlated Normal values: it takes the decorrelation distances and the positions, builds the generators from a preset and draws their phases from the seed. Along a 100 m track, the exponential ACF gives a curve with more variation over short distances than the Gaussian ACF. The truly rough field of the exact exponential ACF would need infinitely many sinusoids; a preset reproduces its shape over the fitted distances, but not the kink at the origin [2, Sec. V]. Both curves start at the same value because they use the same seed and the same number of sinusoids (see "Repeatable simulations"). The values have zero mean and unit variance; a physical quantity is obtained by scaling, e.g. shadow fading in dB with a standard deviation of 6 dB as `6.0 * val`.

In [ ]:
track = zeros(1001, 3); track(:, 1) = linspace(0, 100, 1001);   % 100 m along x, 10 cm steps; [1001, 3]

names = {'Exp256', 'Gauss256'};
figure('Position', [100, 100, 800, 300]); hold on;
for k = 1:2
    val = quadriga_lib.sos_randn(10, track, [], [], names{k}, 7);   % [1001, 1]
    plot(track(:, 1), val, 'LineWidth', 1.2);
end
xlabel('position in m'); ylabel('value'); legend(names); grid on; box on;

In [ ]:
names = {'Exp256', 'Gauss256'};
figure('Position', [100, 100, 1000, 380]);
for k = 1:2
    [f, a, d] = quadriga_lib.sos_preset(names{k}, 10);
    m = quadriga_lib.sos_map(f, quadriga_lib.sos_init_phases(size(f, 1), 1, 3), x, x);   % [400, 400]
    subplot(1, 2, k);
    imagesc(x, x, m); axis xy; axis image; caxis([-3, 3]);
    title(names{k}); xlabel('x in m');
    if k == 1, ylabel('y in m'); end
end
cb = colorbar(); set(cb, 'Position', [0.91, 0.11, 0.015, 0.815]); ylabel(cb, 'value');      % one colorbar for both maps

## Choosing a preset

Presets combine an ACF shape (`Exp`, `Gauss`, `Comb`) with a number of sinusoids (128, 256, 512 or 1024), e.g. `'Comb256'`. More sinusoids approximate the ACF better and cost proportionally more time. The table gives the error of the ACF of each preset against its target, as an rms over distances and 200 directions, inside the fitted range (0 to 5 $d_0$) and beyond it:

| Preset | rms error, 0 to 5 $d_0$ | residual beyond 5 $d_0$ |
|---|---:|---:|
| `Exp128` / `Exp256` / `Exp512` / `Exp1024` | 0.041 / 0.028 / 0.019 / 0.013 | 0.063 / 0.044 / 0.032 / 0.024 |
| `Gauss128` / `Gauss256` / `Gauss512` / `Gauss1024` | 0.023 / 0.017 / 0.009 / 0.005 | 0.062 / 0.043 / 0.031 / 0.023 |
| `Comb128` / `Comb256` / `Comb512` / `Comb1024` | 0.028 / 0.016 / 0.012 / 0.008 | 0.063 / 0.045 / 0.032 / 0.024 |

The residual beyond the fitted range depends only on the number of sinusoids, $1/\sqrt{2N}$. `Comb256` is the default: smooth along a track, the exponential decay of 3GPP at large distances, and a good balance of accuracy and speed. Take `Exp` where the exponential shape is prescribed exactly, a larger preset where the shape of the ACF matters, a smaller one where speed does.

## Checking the correlation

The ACF of a generated field can be estimated from the field itself: take many pairs of positions, group them by their distance, and correlate the values within each group. `sos_acf_estimate` does this for one realization at random positions in a cube around the origin whose size follows the largest distance bin. The cube must span many decorrelation distances, and the estimate of one realization scatters, so the cell averages 20 realizations. For comparison it also evaluates the ACF of the generator itself from the formula above, $\frac{1}{N}\sum_n \cos(2\pi\, \mathbf{f}_n \cdot \mathbf{d})$, along the x axis:

In [ ]:
bins = (0:250).';          % distance bins up to 250 m; the positions fill a cube of 0.8 × 250 m = 200 m edge
ph20 = quadriga_lib.sos_init_phases(size(freq, 1), 20, 2);               % 20 realizations; [256, 20]
acf_est_r = zeros(numel(bins), 20); dist_est_r = zeros(numel(bins), 20);   % [bin, realization]
for r = 1:20
    [acf_est_r(:, r), dist_est_r(:, r)] = quadriga_lib.sos_acf_estimate(freq, ph20(:, r), bins, 1000, [], [], r);
end
acf_est = mean(acf_est_r, 2);
dist_est = dist_est_r(:, 1);
ok = isfinite(acf_est);                                                  % bins without enough pairs are NaN
acf_gen_x = mean(cos(2 * pi * dist * freq(:, 1).'), 2);                  % ACF of the generator along x; [200, 1]

figure('Position', [100, 100, 640, 360]); hold on;
plot(dist, acf, 'k--', 'LineWidth', 1.5);
plot(dist, acf_gen_x, 'Color', [0.85, 0.33, 0.10], 'LineWidth', 1.5);
plot(dist_est(ok), acf_est(ok), 'o', 'MarkerSize', 3, 'Color', [0, 0.45, 0.74], 'MarkerFaceColor', [0, 0.45, 0.74]);
xlim([0, 50]); xlabel('distance d in m'); ylabel('ACF'); grid on; box on;
legend('target (Comb)', 'generator (formula)', 'estimated from the field');
fprintf('ACF at 10 m: target %.2f, generator %.2f, estimated %.2f\n', interp1(dist, acf, 10), ...
    interp1(dist, acf_gen_x, 10), interp1(dist_est(ok), acf_est(ok), 10));

The estimate scatters around the ACF of the generator, and the generator follows the target closely; it differs most near $d_0$, by a few hundredths, where the `Comb` shape changes from `Gauss` to `Exp`. With a smaller cube, a few decorrelation distances across, the estimate falls below the target: the correlation within each bin is taken around the mean of that small region, which differs from the mean of the ensemble (see "Repeatable simulations" below). In two dimensions, the ACF of the generator itself can be compared with the target directly: `sos_acf_2d` gives the target on a grid of displacements, `sos_acf_approx` the ACF of the frequencies of a generator. Both are round: the field has the same correlation in every direction.

In [ ]:
[acf_target, d2] = quadriga_lib.sos_acf_2d(acf, dist);   % target on a grid of displacements; [797, 797]
acf_gen = quadriga_lib.sos_acf_approx(freq, dist);          % generator; [797, 797, 3]: x-y, x-z, y-z plane

maps = {acf_target, acf_gen(:, :, 1)};
titles = {'target', 'generator, x-y plane'};
figure('Position', [100, 100, 1000, 380]);
for k = 1:2
    subplot(1, 2, k);
    imagesc(d2, d2, maps{k}); axis xy; axis image; caxis([-0.1, 1]);
    xlim([-30, 30]); ylim([-30, 30]); title(titles{k}); xlabel('dx in m');
    if k == 1, ylabel('dy in m'); end
end
cb = colorbar(); set(cb, 'Position', [0.91, 0.11, 0.015, 0.815]); ylabel(cb, 'ACF');        % one colorbar for both maps

## Repeatable simulations

The seed selects the phases, and the phases select the field. A later call with the same seed, decorrelation distance and preset returns the same field, also at positions that were not part of the first call. A track can therefore be extended without changing the part already computed:

In [ ]:
first = quadriga_lib.sos_randn(10, track(1:501, :), [], [], 'Comb256', 11);   % first 50 m
whole = quadriga_lib.sos_randn(10, track, [], [], 'Comb256', 11);             % all 100 m, in a later call
disp(['first 50 m unchanged: ', mat2str(isequal(first, whole(1:501, :)))])

A seed selects a set of phases, not a field. Two calls with the same seed but different decorrelation distances use the same phases with scaled frequencies: the field for 20 m is the field for 10 m stretched by a factor of two. The two fields are equal at the origin and strongly correlated near it, which is wrong for quantities that must be independent, such as shadow fading and delay spread. One call with several decorrelation distances gives independent generators. The cell measures the correlation over 1000 seeds at four positions:

In [ ]:
p4 = [0, 0, 0; 5, 0, 0; 20, 0, 0; 100, 0, 0];   % positions along x in m; [4, 3]
two_a = zeros(1000, 4); two_b = zeros(1000, 4);   % two calls, 10 m and 20 m; [seed, position]
one_a = zeros(1000, 4); one_b = zeros(1000, 4);   % one call with both distances
for s = 0:999
    two_a(s + 1, :) = quadriga_lib.sos_randn(10, p4, [], [], 'Comb256', s).';
    two_b(s + 1, :) = quadriga_lib.sos_randn(20, p4, [], [], 'Comb256', s).';
    ab = quadriga_lib.sos_randn([10, 20], p4, [], [], 'Comb256', s);                  % [4, 2]
    one_a(s + 1, :) = ab(:, 1).'; one_b(s + 1, :) = ab(:, 2).';
end
for ip = 1:4
    c2 = corrcoef(two_a(:, ip), two_b(:, ip));
    c1 = corrcoef(one_a(:, ip), one_b(:, ip));
    fprintf('x = %5.0f m:  same seed, two calls %+.2f   one call %+.2f\n', p4(ip, 1), c2(1, 2), c1(1, 2));
end

To keep a generator for later, store its frequencies and phases: `sos_preset` gives the frequencies, `sos_init_phases` the phases, and `sos_val` evaluates them at any positions, as in the first sections.

One realization is not the ensemble. The mean over a 100 m track of one realization is not zero; for an exponential ACF with $d_0 = 10$ m it scatters with a standard deviation of about 0.42 around zero. The variance of such local averages of a random field is a classic topic of random-field theory [12]. Many realizations show this:

In [ ]:
f_exp = quadriga_lib.sos_preset('Exp256', 10);
ph = quadriga_lib.sos_init_phases(size(f_exp, 1), 2000, 21);            % 2000 realizations; [256, 2000]
track_mean = mean(quadriga_lib.sos_val(f_exp, ph, track), 1);           % mean of each realization over 100 m; [1, 2000]

figure('Position', [100, 100, 640, 300]);
hist(track_mean, 40);
xlabel('mean over the 100 m track'); ylabel('number of realizations');
fprintf('standard deviation of the track mean: %.2f\n', std(track_mean));

A simulation that averages over the users of one area therefore needs many realizations (drops), or an area that spans many decorrelation distances.

## Uniform and integer values

`sos_rand` gives spatially correlated values that are uniform between 0 and 1, `sos_randi` integers from 1 to a maximum. A threshold on uniform values gives a map of two states: with a threshold of 0.3, each position is "blocked" with a probability of 30 %. This is how spatially consistent line-of-sight states are drawn in the 3GPP channel model [3, clause 7.6.3.3], and how two-phase materials are modeled [7]. Integer values give regions of a few classes:

In [ ]:
[gx, gy] = meshgrid(x, x);
grid_pos = [gx(:), gy(:), zeros(numel(gx), 1)];                                    % all grid points {x, y, z} in m; [160000, 3]

blocked = quadriga_lib.sos_rand(10, grid_pos, [], [], 'Comb256', 3) < 0.3;         % 30 % blocked; [160000, 1]
classes = quadriga_lib.sos_randi(10, grid_pos, 4, [], [], 'Comb256', 9);           % classes 1 to 4; [160000, 1]

figure('Position', [100, 100, 1000, 380]);
subplot(1, 2, 1);
image(x, x, repmat(1 - reshape(double(blocked), size(gx)), [1, 1, 3])); axis xy; axis image;   % black = blocked
title(sprintf('blocked: %.0f%% of the area', 100 * mean(blocked))); xlabel('x in m'); ylabel('y in m');
subplot(1, 2, 2);
imagesc(x, x, reshape(classes, size(gx))); axis xy; axis image; caxis([0.5, 4.5]);
cmap = colormap(); colormap(cmap(round(linspace(1, size(cmap, 1), 4)), :));        % four colors of the default map
title('classes 1 to 4'); xlabel('x in m');
cb = colorbar; set(cb, 'YTick', 1:4); ylabel(cb, 'class');

The share of the area in this one map is 33 %, not 30 %, for the same reason as the track mean above: one map spans only 20 decorrelation distances, and the 30 % hold for the ensemble. Over many maps of this size, the share scatters by about 5 percentage points (one standard deviation). The classes of `sos_randi` are value bands of one field, so neighboring regions have neighboring classes.

## Comparison with a matrix method

The Cholesky method from the table above is exact: it builds the covariance matrix of all positions and factors it. In MATLAB this takes a few lines for 400 random positions:

In [ ]:
% 400 random positions {x, y, z} in m; [400, 3]; MATLAB and Octave draw different numbers for the same seed
rng(12); p400 = [60 * rand(400, 2), zeros(400, 1)];
D = sqrt((p400(:, 1) - p400(:, 1).').^2 + (p400(:, 2) - p400(:, 2).').^2);   % distances between all positions; [400, 400]
L = chol(exp(-D / 10), 'lower');                                             % exponential ACF, d0 = 10 m
v_chol = L * randn(400, 1);
v_sos = quadriga_lib.sos_randn(10, p400, [], [], 'Exp256', 12);

vals = {v_chol, v_sos};
titles = {'Cholesky', 'sum of sinusoids (Exp256)'};
figure('Position', [100, 100, 1000, 380]);
for k = 1:2
    subplot(1, 2, k);
    scatter(p400(:, 1), p400(:, 2), 60, vals{k}, 'filled'); caxis([-2.5, 2.5]); axis equal; box on;
    xlim([0, 60]); ylim([0, 60]); title(titles{k}); xlabel('x in m');
    if k == 1, ylabel('y in m'); end
end
cb = colorbar(); set(cb, 'Position', [0.91, 0.11, 0.015, 0.815]); ylabel(cb, 'value');      % one colorbar for both panels
% 8 bytes per double
fprintf('covariance matrix: %.1f MB for 400 positions, %.0f GB for 100 000\n', 8 * numel(D) / 1e6, 8 * 100000^2 / 1e9);

Both fields follow the same target ACF: the Cholesky field exactly at these positions, the sum-of-sinusoids field approximately (it smooths the kink of the exponential ACF at the origin, and its values are only approximately Normal). The main difference is the cost: the matrix grows with the square of the number of positions, its factorization with the cube [9, Sec. 2.1], and every new position needs a new factorization. The sum-of-sinusoids generator needs only its frequencies and phases and evaluates any new position directly, which makes it suitable for long tracks, large areas and many users.

## Summary

- A random field is described by its ACF; the decorrelation distance $d_0$ is where the ACF falls to $1/e$.
- The sum-of-sinusoids method adds sinusoids with fitted frequencies and random phases. It needs no grid, little memory, and evaluates any position at any time.
- Presets combine an ACF shape with a number of sinusoids; `Comb256` is the default.
- A seed fixes the phases and makes a simulation repeatable; independent quantities need independent generators.
- One realization is not the ensemble: averages need many realizations.

The next tutorial shows how to fit the frequencies to an ACF of your own, and where the method reaches its limits.

## References

[1] C. E. Rasmussen, C. K. I. Williams, *Gaussian Processes for Machine Learning*, MIT Press, 2006, chapters 2 and 4. [Free online](https://gaussianprocess.org/gpml/chapters/).

[2] S. Jaeckel, L. Raschkowski, F. Burkhardt, L. Thiele, "Efficient sum-of-sinusoids-based spatial consistency for the 3GPP new-radio channel model", IEEE Globecom Workshops, 2018. [PDF](publications/2018_sum_of_sinusoids.pdf), [doi:10.1109/GLOCOMW.2018.8644265](https://doi.org/10.1109/GLOCOMW.2018.8644265).

[3] 3GPP TR 38.901 V19.5.0 (2026-09), "Study on channel model for frequencies from 0.5 to 100 GHz", clauses 7.4.4 (exponential ACF of the shadow fading), 7.5 (Table 7.5-6, correlation distances) and 7.6.3 (spatial consistency). [Online](https://www.3gpp.org/ftp/Specs/archive/38_series/38.901/).

[4] S. Müller, L. Schüler, A. Zech, F. Heße, "GSTools v1.3: a toolbox for geostatistical modelling in Python", Geosci. Model Dev. 15, 3161-3182, 2022. [doi:10.5194/gmd-15-3161-2022](https://doi.org/10.5194/gmd-15-3161-2022).

[5] J. Tessendorf, "Simulating ocean water", course notes, 1999-2004. [PDF](https://jtessen.people.clemson.edu/reports/papers_files/coursenotes2004.pdf).

[6] M. Charnotskii, "Four methods for generation of turbulent phase screens: comparison", 2019. [arXiv:1911.09185](https://arxiv.org/abs/1911.09185). Published as "Comparison of four techniques for turbulent phase screens simulation", J. Opt. Soc. Am. A 37(5), 738, 2020, [doi:10.1364/JOSAA.385754](https://doi.org/10.1364/JOSAA.385754).

[7] A. P. Roberts, M. Teubner, "Transport properties of heterogeneous materials derived from Gaussian random fields: bounds and simulation", Phys. Rev. E 51, 4141-4154, 1995. [doi:10.1103/PhysRevE.51.4141](https://doi.org/10.1103/PhysRevE.51.4141).

[8] M. Gudmundson, "Correlation model for shadow fading in mobile radio systems", Electronics Letters 27(23), 2145-2146, 1991. [doi:10.1049/el:19911328](https://doi.org/10.1049/el:19911328).

[9] D. P. Kroese, Z. I. Botev, "Spatial process generation", 2013. [arXiv:1308.0399](https://arxiv.org/abs/1308.0399). Published as "Spatial process simulation", Lecture Notes in Mathematics 2120, Springer, 2015, pp. 369-404, [doi:10.1007/978-3-319-10064-7_12](https://doi.org/10.1007/978-3-319-10064-7_12).

[10] M. Shinozuka, G. Deodatis, "Simulation of stochastic processes by spectral representation", Appl. Mech. Rev. 44(4), 191-204, 1991. [doi:10.1115/1.3119501](https://doi.org/10.1115/1.3119501).

[11] S. O. Rice, "Mathematical analysis of random noise" (Parts I and II), Bell Syst. Tech. J. 23(3), 282-332, 1944. [doi:10.1002/j.1538-7305.1944.tb00874.x](https://doi.org/10.1002/j.1538-7305.1944.tb00874.x).

[12] E. Vanmarcke, *Random Fields: Analysis and Synthesis*, revised edition, World Scientific, 2010. [doi:10.1142/5807](https://doi.org/10.1142/5807).

[13] GSTools documentation, covariance models (length scale and integral scale). [Online](https://geostat-framework.readthedocs.io/projects/gstools/en/stable/api/gstools.covmodel.Gaussian.html).

[14] Z. Wang, E. K. Tameh, A. R. Nix, "A sum-of-sinusoids based simulation model for the joint shadowing process in urban peer-to-peer radio channels", IEEE VTC 2005-Fall, vol. 3, pp. 1732-1736. [doi:10.1109/VETECF.2005.1558401](https://doi.org/10.1109/VETECF.2005.1558401).

For further reading: [1] chapter 4 for the theory of ACFs and spectra; M. Pätzold, *Mobile Radio Channels*, 2nd ed., Wiley, 2011, for sum-of-sinusoids models in radio channels; [10] for the spectral representation method in engineering.